# Module 02-2: 조건부 엣지 그래프 - 점수 기반 분기 (솔루션)

---
> 참고 문서: `docs/part1-foundations/02-langgraph-fundamentals.md`

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("환경 설정 완료")

## 실습 1: ScoreState 정의 (솔루션)

In [ ]:
# 솔루션
class ScoreState(TypedDict):
    score: int
    grade: str
    message: str

print("ScoreState 정의 완료")

In [ ]:
# 검증 셀
assert 'ScoreState' in dir(), "ScoreState가 정의되어야 합니다"
test: ScoreState = {"score": 85, "grade": "", "message": ""}
assert test["score"] == 85, "score 필드가 올바르게 동작해야 합니다"
print("✅ 실습 1 완료!")

## 실습 2: 노드 함수 3개 구현 (솔루션)

In [ ]:
# 솔루션
def check_score(state: ScoreState) -> dict:
    """점수를 확인하고 합격/불합격 등급을 설정하는 노드"""
    grade = "합격" if state["score"] >= 80 else "불합격"
    return {"grade": grade}


def congratulate(state: ScoreState) -> dict:
    """합격 축하 메시지를 생성하는 노드"""
    return {"message": f"축하합니다! {state['score']}점으로 합격하셨습니다."}


def encourage(state: ScoreState) -> dict:
    """격려 메시지를 생성하는 노드"""
    return {"message": f"{state['score']}점입니다. 다음 기회에 도전해보세요!"}


print("노드 함수 구현 완료")

In [ ]:
# 검증 셀
s1 = {"score": 90, "grade": "", "message": ""}
r1 = check_score(s1)
assert r1["grade"] == "합격", "90점은 합격이어야 합니다"

s2 = {"score": 70, "grade": "", "message": ""}
r2 = check_score(s2)
assert r2["grade"] == "불합격", "70점은 불합격이어야 합니다"

s1["grade"] = "합격"
r_congrat = congratulate(s1)
assert "축하" in r_congrat["message"], "congratulate 메시지에 '축하'가 포함되어야 합니다"
assert "90" in r_congrat["message"], "congratulate 메시지에 점수가 포함되어야 합니다"

s2["grade"] = "불합격"
r_encour = encourage(s2)
assert "70" in r_encour["message"], "encourage 메시지에 점수가 포함되어야 합니다"
print("✅ 실습 2 완료!")

## 실습 3: 라우팅 함수 구현 (솔루션)

In [ ]:
# 솔루션
def route_by_grade(state: ScoreState) -> str:
    """등급에 따라 다음 노드를 결정하는 라우팅 함수"""
    if state["grade"] == "합격":
        return "pass"
    return "fail"


print("라우팅 함수 구현 완료")

In [ ]:
# 검증 셀
assert route_by_grade({"score": 90, "grade": "합격", "message": ""}) == "pass", "합격 상태는 'pass'를 반환해야 합니다"
assert route_by_grade({"score": 70, "grade": "불합격", "message": ""}) == "fail", "불합격 상태는 'fail'을 반환해야 합니다"
print("✅ 실습 3 완료!")

## 실습 4: 조건부 엣지 그래프 조립 및 실행 (솔루션)

In [ ]:
# 솔루션
graph = StateGraph(ScoreState)

# 노드 추가
graph.add_node("check_score", check_score)
graph.add_node("congratulate", congratulate)
graph.add_node("encourage", encourage)

# 시작 엣지
graph.add_edge(START, "check_score")

# 조건부 엣지
graph.add_conditional_edges(
    "check_score",
    route_by_grade,
    {
        "pass": "congratulate",
        "fail": "encourage"
    }
)

# 종료 엣지
graph.add_edge("congratulate", END)
graph.add_edge("encourage", END)

# 컴파일
app = graph.compile()

print("조건부 그래프 조립 완료")

In [ ]:
# 검증 셀
result_pass = app.invoke({"score": 90, "grade": "", "message": ""})
assert result_pass["grade"] == "합격", "90점은 합격 등급이어야 합니다"
assert "축하" in result_pass["message"], "합격 시 축하 메시지가 있어야 합니다"

result_fail = app.invoke({"score": 55, "grade": "", "message": ""})
assert result_fail["grade"] == "불합격", "55점은 불합격 등급이어야 합니다"
assert "55" in result_fail["message"], "불합격 메시지에 점수가 포함되어야 합니다"

print(f"합격(90점): {result_pass['message']}")
print(f"불합격(55점): {result_fail['message']}")
print("✅ 실습 4 완료!")

## 정리

### 오늘 배운 내용
- `add_conditional_edges(출발노드, 라우팅함수, {경로명: 노드명})`으로 **분기 처리**
- 라우팅 함수는 상태를 받아 **경로 이름(문자열)**을 반환
- 여러 종료 노드는 각각 `END`로 연결